# DC-Ops: Auto-Label DC Images with Grounding DINO + SAM

This notebook runs on Google Colab (T4 GPU) to auto-label data center images.

**Steps:**
1. Upload `dc_images.zip` (from `data/dc_images.zip`)
2. Run all cells
3. Download `dc_labels.zip` with YOLO-seg format labels

**Runtime → Change runtime type → T4 GPU**

In [ ]:
# 1. Install dependencies
!pip install -q groundingdino-py segment-anything-py supervision opencv-python-headless
!pip install -q transformers==4.40.2

In [ ]:
# 2. Download model weights
import os, urllib.request
os.makedirs('weights', exist_ok=True)

if not os.path.exists('weights/groundingdino_swint_ogc.pth'):
    print('Downloading Grounding DINO weights...')
    urllib.request.urlretrieve(
        'https://github.com/IDEA-Research/GroundingDINO/releases/download/v0.1.0-alpha/groundingdino_swint_ogc.pth',
        'weights/groundingdino_swint_ogc.pth'
    )

if not os.path.exists('weights/sam_vit_b_01ec64.pth'):
    print('Downloading SAM ViT-B weights...')
    urllib.request.urlretrieve(
        'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth',
        'weights/sam_vit_b_01ec64.pth'
    )

print('Weights ready.')

In [ ]:
# 3. Upload and extract images
from google.colab import files
import zipfile

print('Upload dc_images.zip...')
uploaded = files.upload()

with zipfile.ZipFile('dc_images.zip', 'r') as z:
    z.extractall('.')

import glob
images = sorted(glob.glob('data/sample_images/*.*'))
print(f'{len(images)} images extracted')

In [ ]:
# 4. Load models on GPU
import torch
import numpy as np
import cv2
from pathlib import Path
from groundingdino.util.inference import load_model, load_image, predict
from segment_anything import sam_model_registry, SamPredictor
import groundingdino

print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0)}')

# Find config
pkg_dir = Path(groundingdino.__file__).parent
config_path = str(next(p for p in pkg_dir.rglob('*.py') if 'GroundingDINO' in p.name and 'SwinT' in p.name))

print('Loading Grounding DINO...')
gdino_model = load_model(config_path, 'weights/groundingdino_swint_ogc.pth')

print('Loading SAM ViT-B...')
sam = sam_model_registry['vit_b'](checkpoint='weights/sam_vit_b_01ec64.pth')
sam = sam.to('cuda')
sam_predictor = SamPredictor(sam)

print('Models loaded on GPU!')

In [ ]:
# 5. Define classes and labeling functions

DC_CLASSES = [
    'server rack',        # 0
    'compute tray',       # 1
    'NVLink switch tray', # 2
    'network switch',     # 3
    'power shelf',        # 4
    'cable',              # 5
    'network port',       # 6
    'LED indicator',      # 7
    'label',              # 8
    'fan',                # 9
    'cooling manifold',   # 10
    'cable cartridge',    # 11
    'power connector',    # 12
    'drive bay',          # 13
    'management port',    # 14
    'DPU',                # 15
]

TEXT_PROMPTS = [
    'server rack. compute tray. server. network switch. cable. network port. LED light. label.',
    'fan. cooling manifold. power supply. drive bay. management port. cable cartridge. power connector. network card.',
]

def match_phrase_to_class(phrase):
    phrase = phrase.lower().strip()
    mappings = [
        ('server rack', 0), ('rack', 0), ('cabinet', 0), ('enclosure', 0),
        ('compute tray', 1), ('gpu tray', 1), ('cpu tray', 1), ('tray', 1),
        ('blade', 1), ('server', 1), ('chassis', 1), ('compute', 1),
        ('nvlink switch', 2), ('nvswitch', 2), ('switch tray', 2),
        ('network switch', 3), ('ethernet switch', 3), ('tor switch', 3),
        ('infiniband', 3), ('switch', 3),
        ('power shelf', 4), ('power supply', 4), ('psu', 4), ('power unit', 4),
        ('cable', 5), ('wire', 5), ('cord', 5), ('fiber', 5), ('fibre', 5),
        ('osfp', 6), ('qsfp', 6), ('ethernet port', 6), ('network port', 6),
        ('port', 6), ('socket', 6),
        ('led', 7), ('indicator', 7), ('status light', 7), ('light', 7),
        ('label', 8), ('tag', 8), ('sticker', 8), ('nameplate', 8),
        ('serial', 8), ('barcode', 8),
        ('fan', 9), ('ventilation', 9), ('vent', 9), ('airflow', 9),
        ('manifold', 10), ('coolant', 10), ('liquid cooling', 10),
        ('cooling', 10), ('cold plate', 10), ('heat sink', 10),
        ('cable cartridge', 11), ('cartridge', 11),
        ('power connector', 12), ('power plug', 12), ('bus bar', 12),
        ('power cable', 12), ('power cord', 12),
        ('drive bay', 13), ('drive slot', 13), ('disk', 13), ('nvme', 13),
        ('hard drive', 13), ('ssd', 13), ('drive', 13), ('storage', 13),
        ('bmc', 14), ('management port', 14), ('serial console', 14),
        ('rj45', 14), ('management', 14), ('ipmi', 14),
        ('dpu', 15), ('bluefield', 15), ('nic', 15), ('network card', 15),
        ('connectx', 15), ('network interface', 15), ('adapter', 15),
    ]
    for key, class_id in mappings:
        if key in phrase:
            return class_id
    return None

def mask_to_polygon(mask):
    contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return []
    largest = max(contours, key=cv2.contourArea)
    if cv2.contourArea(largest) < 100:
        return []
    epsilon = 0.02 * cv2.arcLength(largest, True)
    approx = cv2.approxPolyDP(largest, epsilon, True)
    h, w = mask.shape
    polygon = []
    for point in approx.squeeze():
        polygon.extend([float(point[0]) / w, float(point[1]) / h])
    return polygon

print(f'{len(DC_CLASSES)} classes defined')

In [ ]:
# 6. Run auto-labeling on all images
import time

os.makedirs('labels', exist_ok=True)
os.makedirs('visualized', exist_ok=True)

image_files = sorted(glob.glob('data/sample_images/*.*'))
image_files = [f for f in image_files if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp'))]

stats = {'total': 0, 'labeled': 0, 'objects': 0}
start_time = time.time()

for i, img_path in enumerate(image_files):
    stats['total'] += 1
    stem = Path(img_path).stem
    
    try:
        image_source, image_tensor = load_image(img_path)
        all_labels = []
        
        # Two-pass detection
        for prompt in TEXT_PROMPTS:
            boxes, logits, phrases = predict(
                model=gdino_model, image=image_tensor, caption=prompt,
                box_threshold=0.25, text_threshold=0.2, device='cuda'
            )
            
            if len(boxes) == 0:
                continue
            
            h, w, _ = image_source.shape
            boxes_abs = boxes.clone()
            boxes_abs[:, [0, 2]] *= w
            boxes_abs[:, [1, 3]] *= h
            
            sam_predictor.set_image(image_source)
            
            for box, logit, phrase in zip(boxes_abs, logits, phrases):
                box_np = box.cpu().numpy()
                input_box = np.array([
                    box_np[0] - box_np[2]/2, box_np[1] - box_np[3]/2,
                    box_np[0] + box_np[2]/2, box_np[1] + box_np[3]/2,
                ])
                
                masks, scores, _ = sam_predictor.predict(box=input_box, multimask_output=False)
                mask = masks[0]
                
                class_id = match_phrase_to_class(phrase)
                if class_id is None:
                    continue
                
                polygon = mask_to_polygon(mask)
                if polygon:
                    all_labels.append({'class_id': class_id, 'polygon': polygon, 'confidence': float(logit)})
        
        if all_labels:
            # Write YOLO-seg format label
            with open(f'labels/{stem}.txt', 'w') as f:
                for label in all_labels:
                    coords = ' '.join(f'{v:.6f}' for v in label['polygon'])
                    f.write(f"{label['class_id']} {coords}\n")
            
            # Save visualization
            vis = image_source.copy()
            colors = [(0,255,0),(255,128,0),(0,128,255),(255,0,0),(128,0,255),(0,0,255),
                      (255,255,0),(255,0,255),(0,255,255),(128,128,0),(0,128,128),(128,0,128),
                      (64,255,64),(255,64,64),(64,64,255),(192,192,0)]
            for label in all_labels:
                poly = label['polygon']
                h, w = vis.shape[:2]
                pts = np.array([[int(poly[j]*w), int(poly[j+1]*h)] for j in range(0, len(poly), 2)], dtype=np.int32)
                color = colors[label['class_id'] % 16]
                cv2.polylines(vis, [pts], True, color, 2)
                if len(pts) > 0:
                    cv2.putText(vis, f"{DC_CLASSES[label['class_id']]} {label['confidence']:.2f}",
                                (pts[0][0], max(pts[0][1]-5, 15)), cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1)
            cv2.imwrite(f'visualized/{stem}_labeled.jpg', vis)
            
            stats['labeled'] += 1
            stats['objects'] += len(all_labels)
        
        if (i+1) % 10 == 0:
            elapsed = time.time() - start_time
            per_img = elapsed / (i+1)
            remaining = per_img * (len(image_files) - i - 1)
            print(f'[{i+1}/{len(image_files)}] {stats["objects"]} objects so far | {per_img:.1f}s/img | ~{remaining/60:.0f}min left')
    
    except Exception as e:
        print(f'[{i+1}] {stem}: error - {e}')

elapsed = time.time() - start_time
print(f'\nDone! {stats["labeled"]}/{stats["total"]} images labeled, {stats["objects"]} total objects')
print(f'Time: {elapsed/60:.1f} minutes ({elapsed/stats["total"]:.1f}s per image)')

In [ ]:
# 7. Save classes.txt and zip results
with open('labels/classes.txt', 'w') as f:
    f.write('\n'.join(DC_CLASSES) + '\n')

import json
meta = {'classes': DC_CLASSES, 'stats': stats}
with open('labels/meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

# Zip labels
!zip -r dc_labels.zip labels/
!zip -r dc_visualized.zip visualized/

print(f'Labels: {len(glob.glob("labels/*.txt"))} files')
print('Ready to download!')

In [ ]:
# 8. Download results
from google.colab import files
files.download('dc_labels.zip')
files.download('dc_visualized.zip')

In [ ]:
# 9. (Optional) Preview a few labeled images
import matplotlib.pyplot as plt

vis_files = sorted(glob.glob('visualized/*_labeled.jpg'))[:6]
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
for ax, vf in zip(axes.flat, vis_files):
    img = cv2.imread(vf)
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(Path(vf).stem, fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()